In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
base_path = "s3://sportsbar-dp-child-company-prac/orders/"
landing = f'{base_path}landing/'
processed = f'{base_path}processed/'

In [0]:
%run /Workspace/Users/ashishbudz@gmail.com/databricks_pipeline/1_setup/utilities

In [0]:
# Utilities
dbutils.widgets.text("catalog","fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

In [0]:
# Getting the utilities
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
# Reading data from s3 storage (landing folder)
landing_df = spark.read.options(inferSchema=True, header=True).csv(f'{landing}*.csv').withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")

In [0]:
display(landing_df)

In [0]:
# Sending dataframe to the Bronze layer. (Existing table and staging area)
landing_df.printSchema()

In [0]:
landing_df.write.format("delta").option("delta.enableChangeDataFeed", True).mode("append").saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

In [0]:
landing_df.write.format("delta").option("delta.enableChangeDataFeed", True).mode("append").saveAsTable(f'{catalog}.{bronze_schema}.staging_{data_source}')

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source}")

In [0]:
display(df_bronze.count())

In [0]:
display(df_bronze)

In [0]:
# (Bronze to Silver) Standardizing date
df_bronze = df_bronze.withColumn(
    "order_placement_date", F.regexp_replace(F.column("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

In [0]:
df_bronze = df_bronze.withColumn(
        "order_placement_date",
        F.coalesce(
            F.try_to_date(F.column("order_placement_date"), "yyyy/MM/dd"),
            F.try_to_date(F.column("order_placement_date"), "yyyy-MM-dd"),
            F.try_to_date(F.column("order_placement_date"), "dd/MM/yyyy"),
            F.try_to_date(F.column("order_placement_date"), "dd-MM-yyyy"),
            F.try_to_date(F.column("order_placement_date"), "MMMM dd, yyyy")
        )
)

In [0]:
display(df_bronze)

In [0]:
df_bronze = df_bronze.withColumn(
        "customer_id",
        F.when(F.column("customer_id").rlike("^[0-9]+$"), F.column("customer_id"))
        .otherwise(F.lit("999999"))
    )

In [0]:
display(
    df_bronze.where(F.column("order_qty") < 0)
)

In [0]:
df_bronze = df_bronze.dropna(subset=["order_qty"])

In [0]:
display(df_bronze)

In [0]:
filelist = dbutils.fs.ls(f'{landing}')

In [0]:
for file in filelist:
    dbutils.fs.mv(file.path,f'{processed}{file.name}', True)

In [0]:
# Removing duplicates
df_bronze = df_bronze.dropDuplicates(["order_id","order_placement_date","customer_id","product_id","order_qty"])

In [0]:
# Converting product_id to string
df_bronze = df_bronze.withColumn(
        "product_id",
        F.column("product_id").cast("string")
    )

In [0]:
display(df_bronze.where((F.column("product_id") == "99999999") | (F.column("product_id") == "88888888")))

In [0]:
# Check min date and max date to confirm it's correct
display(
df_bronze.agg(
    F.min(F.column("order_placement_date")).alias("min_date"),
    F.max(F.column("order_placement_date")).alias("max_date")
))

In [0]:
# Merge with Product table to get the product code
product_table = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.products')

In [0]:
display(product_table)

In [0]:
# Joining bronze table with Product table to get the product code
df_bronze = (product_table.join(df_bronze, on="product_id", how="inner").select(df_bronze["*"], product_table["product_code"]))

In [0]:
display(df_bronze)

In [0]:
# Writing data to the Silver Layer, and the Staging area 

if not (spark.catalog.tableExists(f'{catalog}.{silver_schema}.{data_source}')):
    df_bronze.write.format("delta").option("delta.enableChangeDataFeed",True).option("mergeSchema", True).mode("overwrite").saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
else:
    silver = DeltaTable.forName(spark, f'{catalog}.{silver_schema}.{data_source}')
    silver.alias("silver").merge(df_bronze.alias("bronze"), "silver.order_id = bronze.order_id AND silver.order_placement_date = bronze.order_placement_date AND silver.customer_id = bronze.customer_id AND silver.product_id = bronze.product_id AND silver.product_code = bronze.product_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
df_bronze.printSchema()

In [0]:
# Sending the data to the Staging area too alongside the Silver table
df_bronze.write.format("delta").option("delta.enableChangeDataFeed",True).mode("overwrite").saveAsTable(f'{catalog}.{silver_schema}.staging_{data_source}')

In [0]:
df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.staging_{data_source}')

In [0]:
display(df_silver)

In [0]:
df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.staging_{data_source}')

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.select(['order_id','order_placement_date','customer_id','product_code','order_qty'])

In [0]:
df_silver = df_silver.withColumnsRenamed({"order_placement_date":"date","customer_id":"customer_code","order_qty":"sold_quantity"})

In [0]:
display(df_silver)

In [0]:
# Now putting the data into the Gold Layer
if not (spark.catalog.tableExists(f'{catalog}.{gold_schema}.sb_fact_{data_source}')):
    df_silver.write.format("delta").option("delta.enableChangeDataFeed",True).mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.sb_fact_{data_source}")
else:
    gold_table = DeltaTable.forName(spark, f'{catalog}.{gold_schema}.sb_fact_{data_source}')
    gold_table.alias("gold").merge(df_silver.alias("silver"), "gold.order_id = silver.order_id AND gold.date = silver.date AND gold.customer_code = silver.customer_code AND gold.product_code = silver.product_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
# Getting the month Value using with the help of the Silver Staging area
month_value_df = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.staging_{data_source}')

In [0]:
month_value = month_value_df.select(F.trunc(F.column("order_placement_date"), 'MM').alias("start_month"))

In [0]:
month_value = month_value.distinct()

In [0]:
display(month_value)

In [0]:
month_value.createOrReplaceTempView("month_value")

In [0]:
df_gold = spark.sql(f"""
                    SELECT sbf.order_id,sbf.date,sbf.customer_code,sbf.product_code,sbf.sold_quantity FROM fmcg.gold.sb_fact_orders as sbf
                    JOIN month_value as mv 
                    ON trunc(sbf.date,"MM") = mv.start_month
                    """)

In [0]:
display(df_gold)

In [0]:
# Another way of getting the values, based on date condition, from the gold table
# So here we will have 2 data frame, the first will be the whole gold table. The other will be the date dataframe
"""
gold_dataframe = spark.sql(f'SELECT * FROM {catalog}.{gold_schema}.sb_fact_{data_source}')
month_value_gold = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.staging_{data_source}').select(F.trunc("order_placement_date","MM").alias("start_month")).distinct()
"""

In [0]:
"""
gold_filtered_df_1 = gold_dataframe.join(month_value_gold, on= F.trunc(gold_dataframe['date'],"MM") == month_value_gold['start_month'], how= "inner")
"""

In [0]:
#display(gold_filtered_df_1)

In [0]:
# Now the df_gold will have to be Grouped and Aggregated before being sent to the 'Parent' gold table

df_gold = df_gold.groupBy(["date","customer_code","product_code"]).agg(F.sum(F.column("sold_quantity")).alias("sold_quantity"))

In [0]:
# Now merging with the Parent table
gold_parent = DeltaTable.forName(spark,f'{catalog}.{gold_schema}.fact_{data_source}')
gold_parent.alias("parent").merge(df_gold.alias("child"), "parent.date = child.date AND parent.customer_code = child.customer_code AND parent.product_code = child.product_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()